In [1]:
# Engenharia de Atributos — Indicador Criança Alfabetizada

Este notebook aplica, em código, as decisões justificadas na análise
exploratória (`01_eda.ipynb`): quais variáveis entram no modelo, quais são
excluídas por vazamento de dado, e como o conjunto é dividido em treino e
teste sem contaminação entre os dois.

A lógica de transformação vive em `src/preprocessing/` (não só aqui no
notebook) — assim ela é reutilizável no script de treino final, testável,
e não depende de rodar célula por célula na ordem certa pra funcionar.

SyntaxError: invalid decimal literal (1590738880.py, line 4)

In [ ]:
## Carregamento da base já corrigida

Carregamos o parquet consolidado da Fase 2, já com o bug do
`label_alfabetizado` corrigido na origem (ver Fase 2,
`src/gold/run_gold.py`) e reextraído via `src/preprocessing/load_raw_data.py`.

In [2]:
import pandas as pd

df = pd.read_parquet("../data/raw/features_alunos_ml.parquet")
df.shape

(3867999, 15)

In [ ]:
## Construção do dataset de modelagem

`build_dataset()` aplica de uma vez só todas as decisões da EDA:

- Filtra só alunos avaliados (`presenca == "1"`) — exclui o vazamento
  determinístico dos ausentes.
- Remove a rede "Privada" — amostra nacional de apenas 24 alunos, pequena
  demais pra generalizar.
- Cria flags de ausência (`taxa_municipio_ausente`, `meta_ausente`) antes
  de qualquer imputação — a ausência em si é informação territorial, não
  deve ser escondida pela imputação que vem depois.
- Substitui `peso_aluno` ausente por peso neutro (1.0) — não é uma feature,
  é o peso amostral que será usado no treino do modelo (`sample_weight`),
  então tratamos essa lacuna separadamente da imputação de features.

O resultado é um DataFrame só com o que segue para a pipeline: as 5
features preditoras, as 2 flags de ausência, o alvo, o peso amostral e o
`id_municipio` (mantido só como chave de agrupamento do split, não como
feature).

In [3]:
import sys
sys.path.append("..")

from src.preprocessing.build_features import build_dataset

dataset = build_dataset(df)
print("Linhas:", len(dataset))
print("Colunas:", list(dataset.columns))
dataset.isnull().sum()

Linhas: 3355822
Colunas: ['ano', 'sigla_uf', 'rede_label', 'taxa_alfabetizacao_municipio', 'meta_alfabetizacao_municipio_ano', 'taxa_municipio_ausente', 'meta_ausente', 'label_alfabetizado', 'id_municipio', 'peso_aluno']


ano                                       0
sigla_uf                                  0
rede_label                                0
taxa_alfabetizacao_municipio         120858
meta_alfabetizacao_municipio_ano    1589576
taxa_municipio_ausente                    0
meta_ausente                              0
label_alfabetizado                        0
id_municipio                              0
peso_aluno                                0
dtype: int64

In [ ]:
## Split treino/teste por município

O split precisa acontecer **antes** de qualquer imputação ou codificação —
se a mediana usada pra imputar `taxa_alfabetizacao_municipio` fosse
calculada sobre a base inteira (treino + teste), o teste estaria
"vazando" uma estatística sua própria para dentro do treino, ainda que de
forma indireta.

Usamos `GroupShuffleSplit` por `id_municipio`, não um split aleatório por
aluno — decisão direta da investigação de vazamento territorial na EDA
(Hipótese 4): sem isso, o mesmo município apareceria em treino e teste,
e o modelo poderia "decorar" a relação daquele município específico em
vez de generalizar pra municípios nunca vistos.

Validamos com um `assert`: se algum município aparecer nos dois conjuntos
ao mesmo tempo, o código falha ruidosamente em vez de deixar passar um
split incorreto silenciosamente.

In [4]:
from src.preprocessing.split import split_by_municipio

train, test = split_by_municipio(dataset)
print("Treino:", len(train), "linhas,", train["id_municipio"].nunique(), "municípios")
print("Teste:", len(test), "linhas,", test["id_municipio"].nunique(), "municípios")
print("\nBalanceamento do alvo:")
print("Treino:", train["label_alfabetizado"].mean().round(4))
print("Teste:", test["label_alfabetizado"].mean().round(4))

Treino: 2682972 linhas, 4437 municípios
Teste: 672850 linhas, 1110 municípios

Balanceamento do alvo:
Treino: 0.591
Teste: 0.5928


In [5]:
from src.preprocessing.pipeline import build_preprocessor, NUMERIC_WITH_MISSING, NUMERIC_COMPLETE, CATEGORICAL

FEATURE_COLS = NUMERIC_WITH_MISSING + NUMERIC_COMPLETE + CATEGORICAL

preprocessor = build_preprocessor()
X_train_transformed = preprocessor.fit_transform(train[FEATURE_COLS])
X_test_transformed = preprocessor.transform(test[FEATURE_COLS])

print("Shape treino transformado:", X_train_transformed.shape)
print("Shape teste transformado:", X_test_transformed.shape)
print("Nomes das colunas geradas:", preprocessor.get_feature_names_out())

Shape treino transformado: (2682972, 33)
Shape teste transformado: (672850, 33)
Nomes das colunas geradas: ['num_missing__taxa_alfabetizacao_municipio'
 'num_missing__meta_alfabetizacao_municipio_ano' 'num_complete__ano'
 'num_complete__taxa_municipio_ausente' 'num_complete__meta_ausente'
 'categorical__sigla_uf_AC' 'categorical__sigla_uf_AL'
 'categorical__sigla_uf_AM' 'categorical__sigla_uf_AP'
 'categorical__sigla_uf_BA' 'categorical__sigla_uf_CE'
 'categorical__sigla_uf_DF' 'categorical__sigla_uf_ES'
 'categorical__sigla_uf_GO' 'categorical__sigla_uf_MA'
 'categorical__sigla_uf_MG' 'categorical__sigla_uf_MS'
 'categorical__sigla_uf_MT' 'categorical__sigla_uf_PA'
 'categorical__sigla_uf_PB' 'categorical__sigla_uf_PE'
 'categorical__sigla_uf_PI' 'categorical__sigla_uf_PR'
 'categorical__sigla_uf_RJ' 'categorical__sigla_uf_RN'
 'categorical__sigla_uf_RO' 'categorical__sigla_uf_RS'
 'categorical__sigla_uf_SC' 'categorical__sigla_uf_SE'
 'categorical__sigla_uf_SP' 'categorical__sigla_uf